# 🛠️ [Phase 0] 인프라 세팅 및 검증 엔진 구축 (Data Setup & Validation)

## 1. 단계 개요
- **목적:** 대용량 데이터(약 1,070만 건)를 안정적으로 처리하기 위한 ML 파이프라인 설계 및 공정한 모델 평가(Validation) 환경 구축 작업임.
- **환경 설정:** Google Colab (런타임 유형: T4 GPU 사용 권장)

---

## 2. 핵심 작업 및 적용 사유

### ① 대용량 데이터 청킹(Chunking) 기반 데이터 로드
- **작업 내용:** 119개 컬럼 전체를 한 번에 적재하지 않고, 타겟 변수(`clicked`) 단일 컬럼만 50만 건 단위로 분할(Batch)하여 순차적으로 읽어옴.
- **적용 사유:** Colab 무료 환경(RAM 12GB)의 물리적 한계로 인한 세션 다운(OOM, Out Of Memory) 현상을 방지하기 위한 선제적 메모리 최적화 조치임.

### ② 교차 검증을 위한 정답지 분할 (Stratified K-Fold)
- **작업 내용:** 타겟 클래스의 심한 불균형(Imbalance)을 고려하여, 각 폴드(Fold)의 클릭(1) 비율이 동일하도록 데이터를 5개의 그룹으로 분할함. 이후 원본 데이터의 행 인덱스와 폴드 번호를 맵핑한 정보만 `train_fold_map.parquet` 파일로 경량화하여 분리 저장함.
- **적용 사유:** 훈련 시 모델이 평가 데이터를 미리 학습하는 정보 누수(Data Leakage)를 원천 차단하고, 객관적이고 신뢰할 수 있는 교차 검증(CV) 지표를 확보하기 위함임.

### ③ 지연 평가(Lazy Evaluation)를 활용한 데이터 스키마 탐색
- **작업 내용:** Polars 라이브러리의 지연 평가(`scan_parquet`) 기능을 활용하여, 10GB 규모의 데이터를 물리적으로 메모리에 로드하지 않고 메타데이터(Schema) 및 상위 5행의 샘플만 추출하여 확인함.
- **적용 사유:** 범주형, 수치형 변수 및 쉼표(,)로 연결된 고유 시퀀스 피처 `seq`(유저 과거 행동 로그)의 형태를 파악하여, 차후 진행될 탐색적 데이터 분석(EDA)의 방향성을 안전하게 수립하기 위함임.

---

## 3. 결론 및 Next Step
> 대규모 데이터 처리 시 발생하는 메모리 병목 현상을 해결하고, 모델의 성능을 객관적으로 평가할 수 있는 안전한 ML 베이스라인 아키텍처 구축을 완료함. 이를 바탕으로 본격적인 글로벌 EDA 및 베이스라인 모델링 단계로 진입함.

# 1. Polars 기반 환경 세팅 및 데이터 로더 설계

In [ ]:
# 1. 필수 라이브러리 설치 및 임포트

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from google.colab import drive

In [ ]:
# 2. 구글 드라이브 마운트 및 경로 설정
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/CTR_Dacon'
RAW_DATA_DIR = os.path.join(BASE_DIR, '원본데이터')
PROCESSED_DIR = os.path.join(BASE_DIR, '가공데이터')
os.makedirs(PROCESSED_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(RAW_DATA_DIR, 'train.parquet')
FOLD_MAP_PATH = os.path.join(PROCESSED_DIR, 'train_fold_map.parquet')

print("1. 원본 데이터에서 타겟(clicked) 값만 가볍게 추출 중...")
parquet_file = pq.ParquetFile(TRAIN_PATH)
targets = []

1. 원본 데이터에서 타겟(clicked) 값만 가볍게 추출 중...


In [ ]:
# 50만 건씩 쪼개서 타겟값만 리스트에 담기 (메모리 사용 최소화)
for batch in parquet_file.iter_batches(columns=['clicked'], batch_size=500_000):
    targets.extend(batch.to_pandas()['clicked'].tolist())

In [ ]:
# 메모리 다이어트
targets = np.array(targets, dtype=np.int8)

print("2. Stratified K-Fold 인덱스 계산 중...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_array = np.zeros(len(targets), dtype=np.int8)

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    fold_array[val_idx] = fold

print("3. Fold 맵핑 파일 저장 중...")
# 원본 데이터의 행 번호(row_id)와 그에 해당하는 fold 번호만 저장
df_folds = pd.DataFrame({
    'fold': fold_array
})

df_folds.to_parquet(FOLD_MAP_PATH)

print(f"✅ OOM 없이 완벽하게 저장 완료! 맵핑 파일 위치: {FOLD_MAP_PATH}")
print(f"총 데이터 건수: {len(df_folds):,}건")

2. Stratified K-Fold 인덱스 계산 중...
3. Fold 맵핑 파일 저장 중...
✅ OOM 없이 완벽하게 저장 완료! 맵핑 파일 위치: /content/drive/MyDrive/Colab Notebooks/CTR_Dacon/가공데이터/train_fold_map.parquet
총 데이터 건수: 10,704,179건


# 파일 내부 구조 확인

In [ ]:
import polars as pl

In [ ]:
# 1. 환경 세팅
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/CTR_Dacon'
TRAIN_PATH = os.path.join(BASE_DIR, '원본데이터/train.parquet')

# 2. Lazy Loading으로 파일 메타데이터만 연결
lazy_df = pl.scan_parquet(TRAIN_PATH)

# 3. 스키마(Schema) 확인: 컬럼명과 데이터 타입
schema = lazy_df.collect_schema()
print(f"✅ 총 컬럼 개수: {len(schema)}개")

# 119개가 너무 많으니, 어떤 종류의 컬럼들이 있는지 앞부분 15개만 확인해봅니다.
print("\n=== 데이터 스키마 (상위 15개 컬럼) ===")
for col_name, dtype in list(schema.items())[:15]:
    print(f"- {col_name}: {dtype}")

# 4. 상위 5개 행 데이터 샘플링 (DB의 LIMIT 5 와 동일)
print("\n=== 데이터 미리보기 (상위 5행) ===")
# head(5)를 걸고 collect()를 하면 딱 5줄만 메모리에 올라오므로 매우 안전합니다.
display(lazy_df.head(5).collect().to_pandas())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 총 컬럼 개수: 119개

=== 데이터 스키마 (상위 15개 컬럼) ===
- gender: String
- age_group: String
- inventory_id: String
- day_of_week: String
- hour: String
- seq: String
- l_feat_1: Float32
- l_feat_2: Float32
- l_feat_3: Float32
- l_feat_4: Float32
- l_feat_5: Float32
- l_feat_6: Float32
- l_feat_7: Float32
- l_feat_8: Float32
- l_feat_9: Float32

=== 데이터 미리보기 (상위 5행) ===


,gender,age_group,inventory_id,day_of_week,hour,seq,l_feat_1,l_feat_2,l_feat_3,l_feat_4,...,history_b_22,history_b_23,history_b_24,history_b_25,history_b_26,history_b_27,history_b_28,history_b_29,history_b_30,clicked
0,1.0,7.0,36,5,13,"9,18,269,516,57,97,527,74,317,311,269,479,57,7...",1.0,2.0,1.0,23.0,...,0.070092,0.070092,0.011682,0.004673,0.087226,0.049843,0.015576,0.040498,0.051401,0
1,1.0,7.0,2,5,08,"9,144,269,57,516,97,527,74,315,317,311,269,479...",2.0,2.0,3.0,17.0,...,0.072990,0.072990,0.012165,0.004866,0.045416,0.051904,0.016220,0.042172,0.026763,0
2,1.0,7.0,36,5,11,"269,516,57,97,165,527,74,77,317,269,75,450,15,...",1.0,2.0,1.0,7.0,...,0.057177,0.057177,0.009530,0.003812,0.035577,0.081318,0.012706,0.033036,0.062898,0
3,1.0,8.0,37,5,11,"269,57,516,21,214,269,561,214,269,561,247,516,...",2.0,2.0,2.0,7.0,...,0.100449,0.100449,0.016741,0.006697,0.062502,0.071430,0.022322,0.058037,0.073659,0
4,2.0,7.0,37,5,07,"144,269,57,516,35,479,57,516,527,74,77,318,193...",2.0,2.0,3.0,24.0,...,0.064512,0.064512,0.010752,0.004301,0.040141,0.045875,0.014336,0.037274,0.023654,0
